# Bank Transaction Fraud Detection System
### Advanced Machine Learning Pipeline · End-to-End

---

> **Dataset:** `Bank_Transaction_Fraud_Detection.csv`  
> **Records:** 200,000 transactions · **Features:** 24 raw → 32 engineered  
> **Target:** `Is_Fraud` (binary) · **Fraud Rate:** ~5.04%  
> **Models:** Logistic Regression · Random Forest · XGBoost · Threshold Tuning  
> **Techniques:** SMOTE · Cross-Validation · SHAP · Risk Scoring · Feature Engineering

---

In [1]:
# Install any missing packages 
import subprocess, sys

REQUIRED = ["scikit-learn", "xgboost", "imbalanced-learn", "shap",
            "matplotlib", "seaborn", "pandas", "numpy"]

for pkg in REQUIRED:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("  All packages available.")

c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  All packages available.


## Import Libraries

In [2]:
# ── Core imports ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib.pyplot   as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches  as mpatches
import seaborn             as sns

# Sklearn — preprocessing
from sklearn.model_selection   import (train_test_split, StratifiedKFold,
                                        cross_val_score, learning_curve)
from sklearn.preprocessing     import LabelEncoder, StandardScaler, label_binarize
from sklearn.pipeline          import Pipeline

# Sklearn — models
from sklearn.linear_model      import LogisticRegression
from sklearn.ensemble          import RandomForestClassifier

# Sklearn — metrics
from sklearn.metrics           import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score,
    ConfusionMatrixDisplay
)

# XGBoost & SMOTE
from xgboost              import XGBClassifier
from imblearn.over_sampling import SMOTE

# SHAP for interpretability
import shap

# ── Global plot style ─────────────────────────────────────────────────────────
DARK = {
    "bg"     : "#0F1117",
    "panel"  : "#1A1D27",
    "grid"   : "#252836",
    "text"   : "#E8EAF0",
    "sub"    : "#8890A4",
    "blue"   : "#4F8EF7",
    "red"    : "#F7614F",
    "green"  : "#4FD1A5",
    "gold"   : "#F7C84F",
    "purple" : "#A78BFA",
}

plt.rcParams.update({
    "figure.facecolor"  : DARK["bg"],
    "axes.facecolor"    : DARK["panel"],
    "axes.edgecolor"    : DARK["grid"],
    "axes.labelcolor"   : DARK["text"],
    "xtick.color"       : DARK["sub"],
    "ytick.color"       : DARK["sub"],
    "text.color"        : DARK["text"],
    "grid.color"        : DARK["grid"],
    "grid.linestyle"    : "--",
    "grid.alpha"        : 0.5,
    "font.family"       : "DejaVu Sans",
    "legend.facecolor"  : DARK["panel"],
    "legend.edgecolor"  : DARK["grid"],
    "axes.titleweight"  : "bold",
    "axes.titlesize"    : 13,
    "axes.titlecolor"   : DARK["text"],
    "figure.dpi"        : 110,
})

SEED = 42
np.random.seed(SEED)

print(" All imports successful.")
print(f"   NumPy   {np.__version__}")
print(f"   Pandas  {pd.__version__}")
import sklearn; print(f"   Sklearn {sklearn.__version__}")
import xgboost; print(f"   XGBoost {xgboost.__version__}")

 All imports successful.
   NumPy   2.4.3
   Pandas  3.0.1
   Sklearn 1.8.0
   XGBoost 3.2.0


## Data Loading 